# EXTRACT RELEVANT PHENOMENON (the use of toponym) & EXTRACT DATA WITHIN THE SPATIAL EXTENT OF THE USE OF TOPONYM TO CALCULATE THE NON-USE OF TOPONYM ...

    # LOAD PACKAGES AND LIB

In [1]:
import pandas as pd 
#import geopandas as gpd
import pickle 
import fiona
from collections import OrderedDict
from pyproj import Transformer
from geopandas import GeoDataFrame
from shapely.geometry import Point
import networkx as nx
import osmnx as ox
import shapely
import matplotlib.pyplot as plt
import shapefile
from shapely.geometry import MultiLineString, MultiPolygon
from shapely.ops import linemerge
import networkx as nx 

import geopy
import shapely.speedups
shapely.speedups.enable()
import momepy
import ogr 
import json

c:\users\ablanchi\appdata\local\programs\python\python36\lib\site-packages\geopandas\_compat.py:110: UserWarning: The Shapely GEOS version (3.8.0-CAPI-1.13.1 ) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [4]:
# Function : Dilatation


def dilatation_of_3(G, subgraph):
    subgraph_node_indices = list(subgraph.nodes())

    all_neighbors = []
    for node_index in subgraph_node_indices:
            neighbors = nx.all_neighbors(G, node_index)
            all_neighbors.extend(neighbors)

    all_neighbors2 = []       
    for node_index in all_neighbors:
            neighbors = nx.all_neighbors(G, node_index)
            all_neighbors2.extend(neighbors)

    all_neighbors3 = []        
    for node_index in all_neighbors2:
            neighbors = nx.all_neighbors(G, node_index)
            all_neighbors3.extend(neighbors)
    
    al_node_dilat = all_neighbors3 + all_neighbors2 + all_neighbors 
    A = set(al_node_dilat)
    
    subgraph_dilated = G.subgraph(A)
    subgraph_dilated = nx.Graph(subgraph_dilated)
    
    return subgraph_dilated

In [5]:
# Function : Erosion

def erosion_of_3(G, subgraph_dilated):
    subgraph_dilated_node_indices = list(subgraph_dilated.nodes())
    G_node_indices = list(G.nodes())
    len(G_node_indices)
    
    G_node_erosion = []

    for node in G_node_indices:
        if node not in subgraph_dilated_node_indices:
            G_node_erosion.append(node)
            
    all_neighbors = []
    for node_index in G_node_erosion:
            neighbors = nx.all_neighbors(G, node_index)
            all_neighbors.extend(neighbors)

    all_neighbors2 = []       
    for node_index in all_neighbors:
            neighbors = nx.all_neighbors(G, node_index)
            all_neighbors2.extend(neighbors)

    all_neighbors3 = []        
    for node_index in all_neighbors2:
            neighbors = nx.all_neighbors(G, node_index)
            all_neighbors3.extend(neighbors)
            
    all_neighbors4  = []   
    for node_index in all_neighbors3:  
        neighbors = nx.all_neighbors(G, node_index)
        all_neighbors4.extend(neighbors)
             
    al_node_erod = all_neighbors3 + all_neighbors2 + all_neighbors #+ all_neighbors4
    B = set(al_node_erod)
    
    nodes_dilerod = []
    for node in subgraph_dilated_node_indices:
        if node not in B:
            nodes_dilerod.append(node)
    
    nodes_dilerod = set(nodes_dilerod)
    
    subgraph_eroded = G.subgraph(nodes_dilerod)
    subgraph_eroded = nx.Graph(subgraph_eroded)
    
    return subgraph_eroded


    # IMPORT LIXELS FROM NETKDE RESULT 

In [6]:
VILLE = "NICE"

In [7]:
df = pd.read_excel("./VILLE/"+VILLE+"/LIXELS_NETKDE_POSITF.xlsx", engine='openpyxl')
streets = gpd.read_file("./VILLE/"+VILLE+"/LIXELS_VILLE.shp")
streets = streets.to_crs("EPSG:2154")
lixels = streets[['lineID', 'uid', 'geometry']]
lixels1 = lixels.merge(df, on ="lineID")
gdf = GeoDataFrame(lixels1, crs="EPSG:2154", geometry=lixels1.geometry_x)


In [8]:
file = open("./VILLE/ALPES-MARITIMES/1_ADS_RELOCALIZED.pickle", "rb")
ADS_AM = pickle.load(file)
file.close()

    # THE USE OF TOPONYM (POSITIF PHENOMENA) (GDF _LIXELS _ POSITIF)

In [9]:
# Delate some columns

notcol = ('lineID', 'uid_x', 'troncon_de_x', 'geometry_x', 'uid_y',
       'troncon_de_y', 'troncon__1', 'troncon__2', 'troncon__3', 'NOM_count',
       'NOM_unique', 'NOM_empty', 'NOM_filled', 'NOM_min', 'NOM_max',
       'NOM_min_le', 'NOM_max_le', 'NOM_mean_l', 'geometry_y', "uid", 'lineID', 'uid_x', 'trncn_d', 'geometry_x', 'uid_y', 'troncon_de',
       'troncon__1', 'troncon__2', 'troncon__3', 'geometry_y', 'geometry')

In [10]:
# CREATE GRAPH G
G_VILLE = momepy.gdf_to_nx(gdf, approach='primal')

In [11]:
## Extract spatial extent of the use of toponym for each toponym in gdb

for col_name in gdf.columns:
    if col_name not in notcol:
        print(col_name)

        #minimal threshold
        gdf[col_name] = gdf[col_name].astype(float)
        list_id_yes = list(set(list(gdf.loc[gdf[col_name] > ((gdf[col_name].max())/100) ].lineID)))
        gdf[col_name] = gdf.apply(lambda x : x[col_name] if (x['lineID'] in list_id_yes) else 0, axis=1)                
        TOPONYM = gdf.loc[gdf[col_name] > 0.0]         
        G = momepy.gdf_to_nx(TOPONYM, approach='primal')

        #morphological closure
        G_dilated = dilatation_of_3(G_VILLE, G) #Dilation of 3 steps
        G_eroded = erosion_of_3(G_VILLE, G_dilated) #Erosion of 3 steps

        #Extraction of the first connected component of the use of toponym.
        largest_cc = max(nx.connected_components(G_eroded), key=len)
        CC = G_eroded.subgraph(largest_cc)

        #save...
        CC1 = nx.Graph(CC)       
        subgraph_end_nodes, subgraph_end_edges = momepy.nx_to_gdf(CC1)
        
        subgraph_edges_lineID = list(subgraph_end_edges["lineID"])
        gdf[col_name] = gdf.apply(lambda x: x[col_name] if x['lineID'] in subgraph_edges_lineID else 0, axis=1)

acropolis
alban
alphonse karr
andrioli
anglais
antiquaires
antoine
antoine ginestiere
archet
arenas
arenes
ariane
arson
augustin
baie anges
barberis
barla
barthelemy
baumettes
bellet
bellevue
berlioz
besset
blue pearl
bonaparte
bornala
boron
borriglione
bottero
brancolar
californie
cap antibes
cap croix
cap ferrat
cap nice
carabacel
carlone
carras
carre or
cassini
caucade
cessole
chambrun
chateau
chateauneuf
cimiez
clemenceau
collines
costiere
cours saleya
cremat
cyrille
cyrille besset
de
delfino
desambrois
dubouchage
eco vallee
edhec
estienne orves
etoile
eveche
fabron
fac lettres
falicon
ferber
fleurie
fleurs
foch
france
franck pilatte
françois
françois grosso
frederic passy
gairaut
gambetta
garibaldi
georges clemenceau
gioffredo
goiran
gorbella
grande corniche
grosso
henri matisse
henri sappia
henry dunant
horloge
hyper
isidore
jean angely
jean behra
jean medecin
jeannet
joseph garnier
lanterne
lepante
liberation
madeleine
magnan
malaussena
mantega
marceau
marguerite
massena
mathis


In [12]:
#save as...

with open("./VILLE/"+VILLE+"/2_PHENOMENE_POSITIF.pickle", 'wb') as f:
    pickle.dump(gdf, f)

path = gdf.to_csv("./VILLE/"+VILLE+"/2_PHENOMENE_POSITIF.csv")

    #   Extraction of data to calculate the non-use of toponym (netkde negatif) : Extraction of ads within the spatial extent of the use of toponym that don't talk about the toponym with a spatial relationship of situation

In [13]:
# ADS
ADS_AM= ADS_AM.to_crs(epsg=2154)

# CREATE DICT OF EACH TOPONYM WITH XY OF DATA (ADS) SELECTED

DICT = {}
for col_name in gdf.columns:
    if col_name not in notcol:
        print(col_name)
        
        TOPONYM = gdf.loc[gdf[col_name] > 0.0] #.unary_union
        G = momepy.gdf_to_nx(TOPONYM, approach='primal')
        TOPONYM_edges = [G.edges[edge]['geometry'] for edge in G.edges]
        TOPONYM_edges_merged_lines = linemerge(TOPONYM_edges)
        #TOPONYM_BUFFER = TOPONYM_edges_merged_lines.buffer(2)
        #OPTION : BUFFER
        TOPONYM_BUFFER = TOPONYM_edges_merged_lines.buffer(500) ################ SELECT IF MORE 
        t = gpd.GeoDataFrame(index=[0], crs='epsg:2154', geometry=[TOPONYM_BUFFER])
        
        test = ADS_AM.loc[ADS_AM.within(t.loc[0,'geometry'])] # CHOOSE TOPONYM_edges_merged_lines OR t ACCORDING ISSUES

        #PARAMETERS
        ADS_WANT =  test.loc[test['new_cat_rs'] == "IN"] #Choose knowledges = IN
        ADS_WANT_YES =  ADS_WANT.loc[ADS_WANT['topo_3'] == col_name]
        list_index_yes = list(set(list(ADS_WANT_YES['idAnnonce'])))
        ADS_WANT_NO = ADS_WANT.loc[ADS_WANT['topo_3'] != col_name] 
        ADS_WANT_NO = ADS_WANT_NO[~ADS_WANT_NO['idAnnonce'].isin(list_index_yes)]
        list_index_no = list(set(list(ADS_WANT_NO['idAnnonce'])))       

        DICT[col_name] = list_index_no
        print(len(DICT))

acropolis
1
alban
2
alphonse karr
3
andrioli
4
anglais
5
antiquaires
6
antoine
7
antoine ginestiere
8
archet
9
arenas
10
arenes
11
ariane
12
arson
13
augustin
14
baie anges
15
barberis
16
barla
17
barthelemy
18
baumettes
19
bellet
20
bellevue
21
berlioz
22
besset
23
blue pearl
24
bonaparte
25
bornala
26
boron
27
borriglione
28
bottero
29
brancolar
30
californie
31
cap antibes
32
cap croix
33
cap ferrat
34
cap nice
35
carabacel
36
carlone
37
carras
38
carre or
39
cassini
40
caucade
41
cessole
42
chambrun
43
chateau
44
chateauneuf
45
cimiez
46
clemenceau
47
collines
48
costiere
49
cours saleya
50
cremat
51
cyrille
52
cyrille besset
53
de
54
delfino
55
desambrois
56
dubouchage
57
eco vallee
58
edhec
59
estienne orves
60
etoile
61
eveche
62
fabron
63
fac lettres
64
falicon
65
ferber
66
fleurie
67
fleurs
68
foch
69
france
70
franck pilatte
71
françois
72
françois grosso
73
frederic passy
74
gairaut
75
gambetta
76
garibaldi
77
georges clemenceau
78
gioffredo
79
goiran
80
gorbella
81
grande c

In [14]:
# save dict as json format

with open("./VILLE/"+VILLE+"/dict_ads_negatives.json", 'w') as fp:
    json.dump(DICT, fp)